# Processing Data Collected as Part of the __ Corpus

In [1]:
import os
import sys
import pandas as pd
from scipy.spatial import minkowski_distance
from tqdm.notebook import tqdm
import warnings; warnings.filterwarnings('ignore')

In [2]:
data_path = '../data'
files_path = os.path.join(data_path, 'grace_data')
outputs_path = os.path.join(data_path, 'server_ready', 'grace_corpus.parquet')

In [3]:
def grab_all_files(PATH, file_type='.csv'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

In [4]:
fs = grab_all_files(files_path)
len(fs)

70

In [5]:
data = []
for f in tqdm(fs):
    data += [pd.read_csv(f)]
    data[-1] = data[-1].rename(columns={'Speaker': 'speaker', 'Statement': 'raw_text', 'Timestamp': 'timestamp'})
    data[-1]['conversation_id'] = f.split('/')[-1].replace('.csv', '')
    data[-1]['corpus'] = 'grace'
    data[-1]['utterance_no'] = data[-1].index.values
    
data = pd.concat(data, ignore_index=True)

  0%|          | 0/70 [00:00<?, ?it/s]

In [6]:
data.head()

,speaker,timestamp,raw_text,conversation_id,corpus,utterance_no
0,R (M),00:00,Okay. What took a while right?,transcript3020,grace,0
1,L (M),00:06,It did. ((laugh)),transcript3020,grace,1
2,L (M),00:07,What do you think about the weather today?,transcript3020,grace,2
3,R (M),00:09,"Oh, well. I’m born and raised in Socal. Are you?",transcript3020,grace,3
4,L (M),00:13,Oh no. I’m from the bay,transcript3020,grace,4


## Removing text formatting

In [7]:
import re

In [8]:
def corrected_text(text, contraction_replacement_nonce='CCOONNTTRRAACCTTIIOONN'):
    res = re.sub(r'\(\(.*?\)\)', ' ', str(text))
    # res = re.sub(r'\[.*?\]', ' ', res)
    
    # find contractions and preserve them . . .
    contractions = list(re.findall(r"\w+'\w+", res))
    for contraction in set(contractions):
        replacement = contraction.replace("'", contraction_replacement_nonce)
        res = res.replace(contraction, replacement)
    res = re.sub(r"(⌋|⌊|⌉|⌈|\[|\])", '', res)
    res = res.replace(':', '')
    
    # remove numbers in parentheses (times???)
    res = re.sub(r'\(\d\.\d\)', ' ', res)
    
    # remove all other special characters.
    res = re.sub(r'[^\w\s]', ' ', res)
    
    res = re.sub(r'\s+', ' ', res).replace('[', ' ').replace(']', ' ').replace(contraction_replacement_nonce, "'")
    
    return res.strip()

In [9]:
data['text'] = [corrected_text(text) for text in tqdm(data['raw_text'])]

  0%|          | 0/24337 [00:00<?, ?it/s]

In [10]:
data[['text', 'raw_text']].head()

,text,raw_text
0,Okay What took a while right,Okay. What took a while right?
1,It did,It did. ((laugh))
2,What do you think about the weather today,What do you think about the weather today?
3,Oh well I m born and raised in Socal Are you,"Oh, well. I’m born and raised in Socal. Are you?"
4,Oh no I m from the bay,Oh no. I’m from the bay


In [11]:
data.isna().sum()

speaker            480
timestamp          481
raw_text           481
conversation_id      0
corpus               0
utterance_no         0
text                 0
dtype: int64

In [12]:
data.loc[data['speaker'].isna()].head()

,speaker,timestamp,raw_text,conversation_id,corpus,utterance_no,text
23857,NaN,NaN,NaN,transcript6850,grace,519,nan
23858,NaN,NaN,NaN,transcript6850,grace,520,nan
23859,NaN,NaN,NaN,transcript6850,grace,521,nan
23860,NaN,NaN,NaN,transcript6850,grace,522,nan
23861,NaN,NaN,NaN,transcript6850,grace,523,nan


In [13]:
data = data.loc[~data['raw_text'].isna()]

In [14]:
data.isna().sum()

speaker            0
timestamp          0
raw_text           0
conversation_id    0
corpus             0
utterance_no       0
text               0
dtype: int64

## Create Speaker indicators

In [15]:
def speaker_info(x):
    gender = re.findall(r'\(.*?\)', x)[0]
    location = re.sub(r'\s+', '', re.sub(r'\(.*?\)', '', x))
    return gender, location

In [16]:
data[['speaker_loc', 'speaker_gender']] = None

In [17]:
for i in data.index:
    gender, location = speaker_info(data['speaker'].loc[i])
    data['speaker_loc'].loc[i] = location
    data['speaker_gender'].loc[i] = gender

In [18]:
data.head()

,speaker,timestamp,raw_text,conversation_id,corpus,utterance_no,text,speaker_loc,speaker_gender
0,R (M),00:00,Okay. What took a while right?,transcript3020,grace,0,Okay What took a while right,R,(M)
1,L (M),00:06,It did. ((laugh)),transcript3020,grace,1,It did,L,(M)
2,L (M),00:07,What do you think about the weather today?,transcript3020,grace,2,What do you think about the weather today,L,(M)
3,R (M),00:09,"Oh, well. I’m born and raised in Socal. Are you?",transcript3020,grace,3,Oh well I m born and raised in Socal Are you,R,(M)
4,L (M),00:13,Oh no. I’m from the bay,transcript3020,grace,4,Oh no I m from the bay,L,(M)


In [19]:
data['speaker'] = data['conversation_id'] + '-' + data['speaker_loc']

In [20]:
data.head()

,speaker,timestamp,raw_text,conversation_id,corpus,utterance_no,text,speaker_loc,speaker_gender
0,transcript3020-R,00:00,Okay. What took a while right?,transcript3020,grace,0,Okay What took a while right,R,(M)
1,transcript3020-L,00:06,It did. ((laugh)),transcript3020,grace,1,It did,L,(M)
2,transcript3020-L,00:07,What do you think about the weather today?,transcript3020,grace,2,What do you think about the weather today,L,(M)
3,transcript3020-R,00:09,"Oh, well. I’m born and raised in Socal. Are you?",transcript3020,grace,3,Oh well I m born and raised in Socal Are you,R,(M)
4,transcript3020-L,00:13,Oh no. I’m from the bay,transcript3020,grace,4,Oh no I m from the bay,L,(M)


## Create juxtaposed corpus: (x,y) pairs

In [21]:
# meta_data_columns = [col for col in list(data) if ('text' not in col)]
meta_data_columns = [
    'speaker',
    'timestamp',
    'conversation_id',
    'corpus',
    'utterance_no',
    'speaker_loc',
    'speaker_gender'
]
meta_data_columns

['speaker',
 'timestamp',
 'conversation_id',
 'corpus',
 'utterance_no',
 'speaker_loc',
 'speaker_gender']

In [22]:
min_distance = 0
max_turns_apart = 200

In [23]:
corpus = []

In [24]:
import warnings; warnings.filterwarnings("ignore")

for cid in tqdm(data['conversation_id'].unique()):
    sub = data.loc[data['conversation_id'].isin([cid])]
    sub_index = sub.index.values
    
    for i in sub_index:
        if i != sub_index[-1]:
            
            # speaker vs. other
            next_line_no = ( (sub_index > i) & (~sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][min_distance:(max_turns_apart+1)]
            
            d_ = data[meta_data_columns+['text']].loc[i].to_dict()
            
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                
                # x_ values
                d = d_.copy()
                
                # y_ values
                # for col in meta_data_columns:
                #     d['next_'+col] = data[col].loc[li]
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_timestamp'] = data['timestamp'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_speaker_loc'] = data['speaker_loc'].loc[li]
                d['next_speaker_gender'] = data['speaker_gender'].loc[li]
                
                # y text
                d['next_text'] = data['text'].loc[li]
                # d['next_utterance_no'] = data['utterance_no'].loc[li]
                # d['next_timestamp'] = data['timestamp'].loc[li]
                # d['next_utterance_delta_no'] = j
                  
                corpus += [d]
            
            # speaker vs. self 
            next_line_no = ( (sub_index > i) & (sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][min_distance:(max_turns_apart+1)]
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                
                # x_ values
                d = d_.copy()
                
                # y_ values
                # for col in meta_data_columns:
                #     d['next_'+col] = data[col].loc[li]
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_timestamp'] = data['timestamp'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_speaker_loc'] = data['speaker_loc'].loc[li]
                d['next_speaker_gender'] = data['speaker_gender'].loc[li]
                
                # y text
                d['next_text'] = data['text'].loc[li]
                
                corpus += [d]

  0%|          | 0/70 [00:00<?, ?it/s]

In [25]:
data = pd.DataFrame(corpus)
print(data.shape)
data.head()

(4566556, 14)


,speaker,timestamp,conversation_id,corpus,utterance_no,speaker_loc,speaker_gender,text,next_speaker,next_timestamp,next_utterance_no,next_speaker_loc,next_speaker_gender,next_text
0,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:06,1,L,(M),It did
1,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:07,2,L,(M),What do you think about the weather today
2,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:13,4,L,(M),Oh no I m from the bay
3,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:16,6,L,(M),This is bay weather
4,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:24,8,L,(M),yeah


In [26]:
rename_columns = dict()
for col in data.columns:
    if col == 'text':
        rename_columns[col] = 'x_utterance'
    elif col == 'next_text':
        rename_columns[col] = 'y_utterance'
    elif 'next_' in col:
        rename_columns[col]  = col.replace('next', 'y')
    else:
        rename_columns[col] = col

In [27]:
data = data.rename(columns=rename_columns)
data.head()

,speaker,timestamp,conversation_id,corpus,utterance_no,speaker_loc,speaker_gender,x_utterance,y_speaker,y_timestamp,y_utterance_no,y_speaker_loc,y_speaker_gender,y_utterance
0,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:06,1,L,(M),It did
1,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:07,2,L,(M),What do you think about the weather today
2,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:13,4,L,(M),Oh no I m from the bay
3,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:16,6,L,(M),This is bay weather
4,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:24,8,L,(M),yeah


In [28]:
data = data.rename(columns={'speaker':'x_speaker','utterance_no': 'x_turn_id', 'y_utterance_no': 'y_turn_id', 'timestamp': 'x_timestamp', 'speaker_loc': 'x_speaker_loc', 'speaker_gender': 'x_speaker_gender'})
data.head()

,x_speaker,x_timestamp,conversation_id,corpus,x_turn_id,x_speaker_loc,x_speaker_gender,x_utterance,y_speaker,y_timestamp,y_turn_id,y_speaker_loc,y_speaker_gender,y_utterance
0,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:06,1,L,(M),It did
1,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:07,2,L,(M),What do you think about the weather today
2,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:13,4,L,(M),Oh no I m from the bay
3,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:16,6,L,(M),This is bay weather
4,transcript3020-R,00:00,transcript3020,grace,0,R,(M),Okay What took a while right,transcript3020-L,00:24,8,L,(M),yeah


In [29]:
data['x_speaker'] = data['corpus'].astype(str) + '-' + data['x_speaker'].astype(str)

data['y_speaker'] = data['corpus'].astype(str) + '-' + data['y_speaker'].astype(str)

In [30]:
data['self'] = data['x_speaker'] == data['y_speaker']
data = data.sort_values(by=['corpus', 'conversation_id', 'x_turn_id', 'self', 'y_turn_id'])
data.index = range(len(data))

In [31]:
data[['corpus', 'x_utterance', 'y_utterance']].isna().mean()

corpus         0.0
x_utterance    0.0
y_utterance    0.0
dtype: float64

In [32]:
data[['corpus', 'x_utterance', 'x_speaker', 'y_speaker', 'y_utterance', 'x_turn_id', 'y_turn_id']].head()

,corpus,x_utterance,x_speaker,y_speaker,y_utterance,x_turn_id,y_turn_id
0,grace,Okay What took a while right,grace-transcript3020-R,grace-transcript3020-L,It did,0,1
1,grace,Okay What took a while right,grace-transcript3020-R,grace-transcript3020-L,What do you think about the weather today,0,2
2,grace,Okay What took a while right,grace-transcript3020-R,grace-transcript3020-L,Oh no I m from the bay,0,4
3,grace,Okay What took a while right,grace-transcript3020-R,grace-transcript3020-L,This is bay weather,0,6
4,grace,Okay What took a while right,grace-transcript3020-R,grace-transcript3020-L,yeah,0,8


In [33]:
data.shape

(4566556, 15)

## Save outputs for server ops

In [35]:
data.to_parquet(outputs_path, engine='fastparquet', compression='snappy')